# 03 — Protocolo e pré-processamento

Demonstra a unidade canônica de 4 s, cria as tabelas de execução e audita os adaptadores específicos de BirdNET e Perch.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


## Protocolo comum

Payload, codecs, índices e similaridade usam sempre quatro segundos. Para novas segmentações de áudio contínuo, o hop é 1 s e é diferente do hop interno de uma STFT. Os manifests congelados atuais não são resegmentados silenciosamente: ECOSoundSet já é distribuído em clipes de 4 s e o freeze PPA1 conserva `[0,4)` e `[4,8)`. Aplicar o hop de 1 s a esses corpora exigirá um novo freeze auditado.


In [ ]:
import pandas as pd
from codeco.protocol import PROTOCOL, adapter_audit

pd.Series(PROTOCOL.as_dict(), name="valor").to_frame()


In [ ]:
from codeco.protocol import sliding_window_starts
pd.DataFrame({"inícios_em_10_s": sliding_window_starts(10.0)})


## Entradas dos modelos fundacionais

BirdNET recebe o centro de 3 s. Perch recebe o sinal de 4 s reamostrado para 32 kHz e 0,5 s de silêncio em cada lado. O padding acontece depois da reconstrução e não entra no payload.


In [ ]:
audit = pd.Series(adapter_audit(), name="valor").to_frame()
audit


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from codeco.protocol import adapt_for_perch

x = np.ones(PROTOCOL.window_samples, dtype=np.float32)
perch = adapt_for_perch(x)
t = np.arange(len(perch)) / PROTOCOL.perch_sample_rate_hz
plt.figure(figsize=(11, 2.8))
plt.plot(t, perch, color="#356a8a")
plt.axvspan(0, 0.5, color="#ccc", alpha=.5, label="padding")
plt.axvspan(4.5, 5.0, color="#ccc", alpha=.5)
plt.xlabel("tempo (s)"); plt.ylabel("amplitude"); plt.title("Adaptador Perch: 4 s + padding simétrico")
plt.legend(); plt.tight_layout(); plt.show()


## Tabelas de execução

Estas tabelas apontam para o áudio instalado, mas não modificam os manifests. Se o áudio não estiver presente nesta máquina, a auditoria continua válida e a execução pesada fica para o EASNCloud.


In [ ]:
from data import prepare_all_clip_tables
clip_paths = prepare_all_clip_tables()
clip_tables = {name: pd.read_csv(path) for name, path in clip_paths.items()}
pd.DataFrame([
    {"corpus": name, "janelas": len(df), "audio_disponivel": int(df.audio_available.sum()), "arquivo": str(clip_paths[name].relative_to(ROOT))}
    for name, df in clip_tables.items()
])
